In [1]:
import zipfile
import os

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_prep import unzip_csv

# Diabetes prediction

In [2]:
def unzip_csv(zip_file_path, extract_to_folder):
    if not os.path.exists(zip_file_path):
        print(f"File {zip_file_path} not found.")
        return
    
    os.makedirs(extract_to_folder, exist_ok=True)

    with zipfile.ZipFile(zip_file_path, "r") as zip_ref:
        zip_ref.extractall(extract_to_folder)
        
        extracted_files = zip_ref.namelist()
        print(f"Successfully extracted files: {", ".join(extracted_files)}")

In [3]:
zip_files = [
    "data/raw/diabetes_prediction_dataset.zip",
]

for zip_path in zip_files:
    unzip_csv(zip_path, "data/processed/")

Successfully extracted files: diabetes_prediction_dataset.csv


In [4]:
data = pd.read_csv("data/processed/diabetes_prediction_dataset.csv")

In [5]:
data

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0
...,...,...,...,...,...,...,...,...,...
99995,Female,80.0,0,0,No Info,27.32,6.2,90,0
99996,Female,2.0,0,0,No Info,17.37,6.5,100,0
99997,Male,66.0,0,0,former,27.83,5.7,155,0
99998,Female,24.0,0,0,never,35.42,4.0,100,0


Some attributes like age and gender are self-explanatory. However, attributes like HbA1c_level, blood_glucose_level, and even bmi require domain knowledge, which is why we go through some explanations below.

**BMI (Body Mass Index)**

bmi: Body Mass Index. A generally useful metric, calculated as weight (kg) / height (m)$^2$. However, it does not account for age, sex, muscle mass, or body fat percentage.

Standard World Health Organization classification:
- Underweight: < 18.5
- Normal weight: 18.5 – 24.9
- Overweight: 25.0 – 29.9
- Obese: 30.0+

**HbA1c_level (Glycated Hemoglobin)**

Reflects the average blood sugar level over the past 2–3 months, measured as a percentage. Unlike a single glucose reading, it is not affected by short-term fluctuations (e.g. what the person ate that morning), which makes it a more stable diagnostic indicator.

Clinical thresholds:
- Normal: < 5.7%
- Prediabetes: 5.7% – 6.4%
- Diabetes: $\geq$ 6.5%

**blood_glucose_level**

A single snapshot measurement of sugar concentration in the blood at the time of testing, expressed in mg/dL (milligrams of glucose per deciliter of blood) - unlike HbA1c which reflects a longer-term average. Depending on whether the reading was fasting or random, the diagnostic thresholds differ:

Fasting glucose:
- Normal: < 100 mg/dL
- Prediabetes: 100 – 125 mg/dL
- Diabetes: $\geq$ 126 mg/dL

Random (non-fasting) glucose:
- Diabetes: $ \geq $ 200 mg/dL

> Note: this dataset does not specify whether readings are fasting or random, which is a limitation worth mentioning in the analysis.

**smoking_history**

Categorical variable with 6 possible values: `never`, `former`, `current`, 
`not current`, `ever`, and `No Info`.

The `No Info` category is not a smoking status - it indicates missing data (the respondent's smoking history was not recorded). Treating it as its own category, rather than dropping it, avoids losing rows, but it should not be interpreted as "never smoked."

The overlap between `former`, `not current`, and `ever` is somewhat ambiguous in the source data and worth noting as a limitation - these may need to be consolidated during preprocessing.

**hypertension** and **heart_disease**

Both are binary indicators (0 = No, 1 = Yes) representing whether the respondent has been previously diagnosed with high blood pressure or heart disease, respectively. These are included as established comorbidity risk factors for type 2 diabetes - hypertension and cardiovascular disease frequently co-occur with diabetes due to shared risk factors such as obesity and metabolic syndrome.

**diabetes** (target variable)

Binary label (0 = No diabetes, 1 = Diabetes). Note the overlap with the clinical thresholds discussed above - since HbA1c $ \geq $ 6.5% and fasting glucose $ \geq $ 126 mg/dL are themselves diagnostic criteria for diabetes, these two features are likely to be extremely strong (possibly near-deterministic) predictors of the target. This should be flagged explicitly, as models may achieve very high accuracy simply by learning the clinical cutoff rather than genuine risk patterns from lifestyle/demographic features.